In [ ]:
import requests
from bs4 import BeautifulSoup
import json
import time

# ============================================================
# CONFIGURAÇÕES
# ============================================================

BASE_URL = "https://www.letras.mus.br"

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    )
}

# Top 10 gêneros
GENEROS = {
    "sertanejo": "Sertanejo",
    "pop": "Pop",
    "rock": "Rock",
    "mpb": "MPB",
    "funk": "Funk",
    "hip-hop-rap": "Rap",
    "pagode": "Pagode",
    "forro": "Forró",
    "gospelreligioso": "Gospel",
    "reggae": "Reggae"
}

MUSICAS_POR_GENERO = 50
TEMPO_ESPERA = 0.5


# ============================================================
# FAZER REQUISIÇÃO
# ============================================================

def fazer_request(url):
    try:
        resposta = requests.get(
            url,
            headers=HEADERS,
            timeout=15
        )

        if resposta.status_code == 200:
            return resposta

        print(
            f"Erro HTTP {resposta.status_code}: {url}"
        )

    except requests.RequestException as e:
        print(f"Erro ao acessar {url}: {e}")

    return None


# ============================================================
# PEGAR AS 50 MÚSICAS DE UM GÊNERO
# ============================================================

def obter_top_50_genero(slug_genero, nome_genero):

    url_genero = f"{BASE_URL}/mais-acessadas/{slug_genero}/"

    print("\n" + "=" * 70)
    print(f"GÊNERO: {nome_genero}")
    print(f"URL: {url_genero}")
    print("=" * 70)

    resposta = fazer_request(url_genero)

    if not resposta:
        return []

    soup = BeautifulSoup(
        resposta.text,
        "html.parser"
    )

    urls_coletadas = []
    urls_vistas = set()

    # ========================================================
    # PROCURA OS LINKS DAS MÚSICAS
    # ========================================================

    links = soup.find_all("a", href=True)

    for a in links:

        href = a.get("href", "").strip()

        if not href:
            continue

        # Só queremos links internos
        if not href.startswith("/"):
            continue

        # Remove parâmetros
        href = href.split("?")[0]
        href = href.split("#")[0]

        # ====================================================
        # Estrutura esperada:
        #
        # /artista/musica/
        #
        # ====================================================

        partes = href.strip("/").split("/")

        if len(partes) != 2:
            continue

        artista_slug = partes[0]
        musica_slug = partes[1]

        if not artista_slug or not musica_slug:
            continue

        # ====================================================
        # Ignora páginas que não são músicas
        # ====================================================

        paginas_ignoradas = [
            "estilos",
            "generos",
            "mais-acessadas",
            "top-artistas",
            "playlists",
            "novidades",
            "blog",
            "sobre",
            "contato",
            "traducao",
            "significado"
        ]

        if artista_slug.lower() in paginas_ignoradas:
            continue

        url_musica = f"{BASE_URL}{href}"

        if url_musica in urls_vistas:
            continue

        urls_vistas.add(url_musica)

        urls_coletadas.append({
            "letra": url_musica,
            "significado": (
                f"{url_musica.rstrip('/')}/significado.html"
            ),
            "genero_ranking": nome_genero
        })

        # Para exatamente nas 50
        if len(urls_coletadas) >= MUSICAS_POR_GENERO:
            break

    print(
        f"{len(urls_coletadas)} músicas encontradas."
    )

    return urls_coletadas


# ============================================================
# EXTRAIR DADOS DA MÚSICA
# ============================================================

def extrair_dados_musica(
    url_letra,
    url_significado,
    genero_ranking
):

    # ========================================================
    # 1. PÁGINA DA LETRA
    # ========================================================

    resposta = fazer_request(url_letra)

    if not resposta:
        return None

    soup = BeautifulSoup(
        resposta.text,
        "html.parser"
    )

    # ========================================================
    # TÍTULO
    # ========================================================

    titulo_tag = (
        soup.find(
            "h1",
            class_="head-title"
        )
        or soup.find("h1")
    )

    titulo = (
        titulo_tag.get_text(strip=True)
        if titulo_tag
        else ""
    )

# ========================================================
    # TÍTULO E ARTISTA
    # ========================================================

    # Captura o Título (h1)
    titulo_tag = (
        soup.find("div", class_=lambda c: c and "title-primary" in c)
        or soup.find("h1")
    )
    titulo = titulo_tag.get_text(strip=True) if titulo_tag else ""

    # Captura o Artista (a.title-secondary ou h2 dentro de title-secondary)
    artista_tag = (
        soup.find("a", class_=lambda c: c and "title-secondary" in c)
        or soup.find("div", class_=lambda c: c and "title-secondary" in c)
    )

    if artista_tag:
        artista = artista_tag.get_text(strip=True)
    else:
        # Fallback para o h2 mais próximo da tag de título
        h2_tag = soup.find("h2")
        artista = h2_tag.get_text(strip=True) if h2_tag else ""
    # ========================================================
    # GÊNEROS
    # ========================================================

    generos = []

    breadcrumbs = soup.find_all(
        ["span", "a"],
        itemprop="name"
    )

    for item in breadcrumbs:

        texto = item.get_text(strip=True)

        if not texto:
            continue

        texto_lower = texto.lower()

        ignorar = [
            titulo.lower(),
            artista.lower(),
            "página inicial",
            "letras.mus.br"
        ]

        if texto_lower not in ignorar:
            generos.append(texto)

    # Remove duplicados
    generos = list(dict.fromkeys(generos))

    # Garante que o gênero do ranking esteja presente
    if genero_ranking not in generos:
        generos.insert(0, genero_ranking)

    # ========================================================
    # LETRA
    # ========================================================

    div_lyric = (
        soup.find(
            "div",
            class_="lyric-original"
        )
        or soup.find(
            "div",
            class_="lyric"
        )
    )

    letra = ""

    if div_lyric:

        paragrafos = []

        for p in div_lyric.find_all("p"):

            texto = p.get_text(
                separator="\n",
                strip=True
            )

            if texto:
                paragrafos.append(texto)

        letra = "\n\n".join(paragrafos)

    # ========================================================
    # VISUALIZAÇÕES
    # ========================================================

    # Busca o container das exibições pela classe "head-info-exib"
    views_container = soup.find("div", class_=lambda c: c and "head-info-exib" in c)

    exibicoes = ""

    if views_container:
        # Pega o número dentro da tag <b> se ela existir
        b_tag = views_container.find("b")
        if b_tag:
            exibicoes = b_tag.get_text(strip=True)
        else:
            # Caso não tenha <b>, pega todo o texto do container
            exibicoes = views_container.get_text(strip=True)
    else:
        # Fallback para classes antigas do site
        views_tag = (
            soup.find("div", class_="cnt-info_exib")
            or soup.find("span", class_="js-exib-count")
        )
        if views_tag:
            exibicoes = views_tag.get_text(strip=True)
    # ========================================================
    # 2. SIGNIFICADO
    # ========================================================

    significados = []

    resposta_significado = fazer_request(
        url_significado
    )

    if resposta_significado:

        soup_sig = BeautifulSoup(
            resposta_significado.text,
            "html.parser"
        )

        div_meaning = soup_sig.find(
            "div",
            class_="lyric-meaning"
        )

        if div_meaning:

            paragrafos_sig = div_meaning.find_all("p")

            if paragrafos_sig:

                significados = [
                    p.get_text(strip=True)
                    for p in paragrafos_sig
                    if p.get_text(strip=True)
                ]

            else:

                texto = div_meaning.get_text(
                    strip=True
                )

                if texto:
                    significados = [texto]

    # ========================================================
    # RETORNO
    # ========================================================

    return {
        "titulo": titulo,
        "artista": artista,
        "metadados": {
            "generos": generos,
            "genero_ranking": genero_ranking,
            "exibicoes": exibicoes,
            "url_letra": url_letra,
            "url_significado": url_significado
        },
        "letra": letra,
        "significados": significados
    }


# ============================================================
# EXECUÇÃO PRINCIPAL
# ============================================================

print("=" * 70)
print("COLETA DE MÚSICAS POR GÊNERO")
print("=" * 70)

todas_musicas = []


# ============================================================
# 1. COLETAR 50 MÚSICAS DE CADA GÊNERO
# ============================================================

for slug_genero, nome_genero in GENEROS.items():

    musicas = obter_top_50_genero(
        slug_genero,
        nome_genero
    )

    todas_musicas.extend(musicas)

    time.sleep(TEMPO_ESPERA)


print("\n" + "=" * 70)
print(
    f"Total de músicas coletadas: "
    f"{len(todas_musicas)}"
)
print("=" * 70)


# ============================================================
# 2. REMOVER MÚSICAS DUPLICADAS
# ============================================================

musicas_unicas = {}

for musica in todas_musicas:

    url = musica["letra"]

    if url not in musicas_unicas:
        musicas_unicas[url] = musica

lista_para_raspar = list(
    musicas_unicas.values()
)

print(
    f"Músicas após remover duplicatas: "
    f"{len(lista_para_raspar)}"
)


# ============================================================
# 3. EXTRAIR DADOS
# ============================================================

base_de_dados = []

total = len(lista_para_raspar)

for i, item in enumerate(
    lista_para_raspar,
    start=1
):

    print(
        f"\n[{i}/{total}] "
        f"{item['genero_ranking']} | "
        f"{item['letra']}"
    )

    dados = extrair_dados_musica(
        item["letra"],
        item["significado"],
        item["genero_ranking"]
    )

    if dados:

        base_de_dados.append(dados)

        print(
            f"OK: {dados['artista']} - "
            f"{dados['titulo']}"
        )

    else:

        print("Falha na extração.")

    time.sleep(TEMPO_ESPERA)


# ============================================================
# 4. SALVAR JSON
# ============================================================

nome_arquivo = (
    "dataset_top10_generos_50_musicas.json"
)

with open(
    nome_arquivo,
    "w",
    encoding="utf-8"
) as arquivo:

    json.dump(
        base_de_dados,
        arquivo,
        ensure_ascii=False,
        indent=4
    )


# ============================================================
# 5. RESUMO FINAL
# ============================================================

print("\n" + "=" * 70)
print("EXTRAÇÃO CONCLUÍDA")
print("=" * 70)

print(
    f"Músicas salvas: {len(base_de_dados)}"
)

print(
    f"Arquivo: {nome_arquivo}"
)

print("=" * 70)

COLETA DE MÚSICAS POR GÊNERO

GÊNERO: Sertanejo
URL: https://www.letras.mus.br/mais-acessadas/sertanejo/
50 músicas encontradas.

GÊNERO: Pop
URL: https://www.letras.mus.br/mais-acessadas/pop/
50 músicas encontradas.

GÊNERO: Rock
URL: https://www.letras.mus.br/mais-acessadas/rock/
50 músicas encontradas.

GÊNERO: MPB
URL: https://www.letras.mus.br/mais-acessadas/mpb/
50 músicas encontradas.

GÊNERO: Funk
URL: https://www.letras.mus.br/mais-acessadas/funk/
50 músicas encontradas.

GÊNERO: Rap
URL: https://www.letras.mus.br/mais-acessadas/hip-hop-rap/
50 músicas encontradas.

GÊNERO: Pagode
URL: https://www.letras.mus.br/mais-acessadas/pagode/
50 músicas encontradas.

GÊNERO: Forró
URL: https://www.letras.mus.br/mais-acessadas/forro/
50 músicas encontradas.

GÊNERO: Gospel
URL: https://www.letras.mus.br/mais-acessadas/gospelreligioso/
50 músicas encontradas.

GÊNERO: Reggae
URL: https://www.letras.mus.br/mais-acessadas/reggae/
50 músicas encontradas.

Total de músicas coletadas: 500
Mús